# Промпт #22 — Prompt Chaining edge case (сломанный шаг 1)

**Техника:** Prompt Chaining  
**Задача:** Что происходит если шаг 1 даёт плохой вывод?  
**Сложность:** ⭐⭐⭐⭐☆

In [1]:
import sys
sys.path.append('..')
from config import get_completion, get_completion_messages

In [2]:
text = """Искусственный интеллект меняет рынок труда быстрее чем ожидалось. 
По данным McKinsey, к 2030 году автоматизация затронет до 30% рабочих мест. 
При этом создаются новые профессии — специалисты по AI, prompt-инженеры, 
аналитики данных. Эксперты считают что ключевой навык будущего — умение 
работать вместе с AI, а не конкурировать с ним. Компании которые внедряют 
AI показывают рост продуктивности на 40% по сравнению с конкурентами."""

# Шаг 1 — намеренно плохой промпт: просим слишком много и без структуры
messages = [
    {"role": "user", "content": f"""Проанализируй текст, выдели всё важное, 
расскажи о чём он, какие есть мнения, факты, цифры, выводы и вообще всё 
что ты думаешь по этой теме.

Текст: {text}"""}
]

step1 = get_completion_messages(messages)
print("=== ШАГ 1: Плохой вывод ===")
print(step1)

# Шаг 2 — тот же запрос что в #21
messages.append({"role": "assistant", "content": step1})
messages.append({"role": "user", "content": """На основе этих тезисов напиши 
резюме в 2 предложения для руководителя который не читал текст."""})

step2 = get_completion_messages(messages)
print("\n=== ШАГ 2: Резюме на основе плохого вывода ===")
print(step2)

# Сравнение
print("\n=== ВЫВОД ===")
print(f"Длина вывода шага 1: {len(step1.split())} слов")
print(f"Длина резюме шага 2: {len(step2.split())} слов")

=== ШАГ 1: Плохой вывод ===
**Анализ текста**

Текст посвящен теме искусственного интеллекта (ИИ) и его влиянию на рынок труда. Основная идея текста заключается в том, что ИИ меняет рынок труда быстрее, чем ожидалось, и что это изменение требует новых навыков и подходов от работников.

**Важные факты и цифры**

* По данным McKinsey, к 2030 году автоматизация затронет до 30% рабочих мест.
* Компании, которые внедряют ИИ, показывают рост продуктивности на 40% по сравнению с конкурентами.

**Мнения и выводы**

* Эксперты считают, что ключевой навык будущего — умение работать вместе с ИИ, а не конкурировать с ним.
* Текст подразумевает, что внедрение ИИ на рабочем месте может привести к созданию новых профессий, таких как специалисты по ИИ, prompt-инженеры и аналитики данных.

**Анализ темы**

Тема ИИ и его влияния на рынок труда является актуальной и важной. Рост автоматизации и ИИ может привести к значительным изменениям на рынке труда, что требует от работников новых навыков и подходов.

## Оценка: 4/5

## Инсайт
Ожидалось что плохой шаг 1 сломает шаг 2 — но резюме получилось читаемым.
Шаг 1 выдал 287 слов вместо чистого списка тезисов, но факты остались верными.
Шаг 2 справился — вытащил суть даже из многословного вывода.

Почему сработало: текст был простым и фактов мало. Модель не могла 
галлюцинировать — все данные (30%, 40%, 2030) были в исходнике.

Когда chaining реально ломается (урок из туториала):
- Шаг 1 галлюцинирует несуществующие факты → шаг 2 строит резюме на выдумке
- Шаг 1 выдаёт неструктурированный вывод для задачи где нужен JSON → шаг 2 падает

Главный вывод: качество chaining зависит от сложности задачи и качества данных.
На простых текстах модель прощает плохой промпт шага 1. На сложных — нет.
Чем критичнее точность, тем важнее делать шаг 1 чистым и предсказуемым.